In [ ]:
import os
import email
import uuid

from dotenv import load_dotenv

load_dotenv()

# from google.colab import userdata
KAGGLE_USERNAME = os.getenv('KAGGLE_USERNAME')
KAGGLE_KEY = os.getenv('KAGGLE_KEY')

import kaggle as kg
import pandas as pd
import numpy as np

from agentic_text_cls import AgenticTextCls
from settings import ModelSettings, AgentSettings
from logging_config import setup_logging, logger

In [ ]:
DATASET_NAME = "wcukierski/enron-email-dataset"
N_DOCUMENTS = 200

In [ ]:
kg.api.dataset_download_files(dataset=DATASET_NAME, path='data', unzip=True)
df_emails = pd.read_csv('./data/emails.csv')
df_emails.info()

In [ ]:
# Helper functions
def get_text_from_email(msg):
    '''To get the content from email objects'''
    parts = []
    for part in msg.walk():
        if part.get_content_type() == 'text/plain':
            parts.append( part.get_payload() )
    return ''.join(parts)

def split_email_addresses(line):
    '''To separate multiple email addresses'''
    if line:
        addrs = line.split(',')
        addrs = frozenset(map(lambda x: x.strip(), addrs))
    else:
        addrs = None
    return addrs

# Parse the emails into a list email objects
messages = list(map(email.message_from_string, df_emails['message']))
df_emails.drop('message', axis=1, inplace=True)
# Get fields from parsed email objects
keys = messages[0].keys()
for key in keys:
    df_emails[key] = [doc[key] for doc in messages]
# Parse content from emails
df_emails['content'] = list(map(get_text_from_email, messages))
# Split multiple email addresses
df_emails['From'] = df_emails['From'].map(split_email_addresses)
df_emails['To'] = df_emails['To'].map(split_email_addresses)

# Extract the root of 'file' as 'user'
df_emails['user'] = df_emails['file'].map(lambda x:x.split('/')[0])
del messages

df_emails.head()

In [ ]:
df_emails.columns

In [ ]:
# Helper function to prepare input for classification
def preprare_input_text(from_field: str, to_field, subject: str, content: str) -> str:
    return f"""From: {from_field}\nTo: {to_field}\nSubject: {subject}\n
    {content}
    """

sample_msg = df_emails[["From", "To", "Subject", "content"]].sample(N_DOCUMENTS, random_state=42).values.tolist()
input_test_list = []
for msg in sample_msg:
    from_field, to_field, subject, content = msg[0], msg[1], msg[2], msg[3]
    from_field = "".join([i for i in from_field])
    to_field = ";".join([i for i in to_field]) if to_field is not None else ""
    input_text = preprare_input_text(from_field=from_field, to_field=to_field, subject=subject, content=content)
    input_test_list.append(input_text)

## Run Agent

In [ ]:
agentic_config = AgentSettings()
setup_logging(agentic_config)
configs = ModelSettings()
ac = AgenticTextCls(
    model_settings=configs, logger=logger, generate_new_metadata_idx=True
)

In [ ]:
for idx_txt, text in enumerate(input_test_list):
    print(f"Processing: {idx_txt} ...")
    ac.classify_text(text=text)

logger.info(f"Dict with topics:\n{ac.topics}")

In [ ]:
# Store results
import json

with open("./output.json", "w") as f:
    json.dump(ac.topics, f)

In [ ]:
len(ac.topics)